In [ ]:
import joblib
import pandas as pd


# Inisiasi Load Model dari Modeling 1
model_m1 = joblib.load('model_modeling_1.pkl')

# inisiasi Load Model
model_inf = joblib.load('best_ecommerce_model_v2.pkl')


# Inisiasi Load Model Final
final_model = joblib.load('final_ecommerce_model_tuned.pkl')

# Inisiasi Data Baru dari Tim Marketing
inf_data = pd.DataFrame([
    {
        'latest_ecommerce_progress': 5, 'bounces': 0, 'time_on_site': 2489, 
        'pageviews': 30, 'channelGrouping': 'Organic Search', 
        'deviceCategory': 'desktop', 'country': 'Canada'
    },
    {
        'latest_ecommerce_progress': 4, 'bounces': 0, 'time_on_site': 436, 
        'pageviews': 50, 'channelGrouping': 'Organic Search', 
        'deviceCategory': 'desktop', 'country': None # Kasus null value
    }
])


# 3. Prediksi
pred_m1 = model_m1.predict(inf_data)
proba_m1 = model_m1.predict_proba(inf_data)[:, 1]

# Inisiasi untuk Tampilkan Hasil
print("=== Hasil Prediksi Modeling 1 ===")
for i, pred in enumerate(pred_m1):
    status = "Akan Membeli" if pred == 1 else "Tidak Membeli"
    print(f"User {i+1}: {status} (Probabilitas: {proba_m1[i]:.2f})")
print("\n==============================")


# inisiasi fungsi Mapping Continent (Wajib sama dengan saat training)
def country_to_continent(country):
    if pd.isna(country) or country == '--null-value--':
        return 'Others'
    
    mapping = {
        'United States': 'North America', 'Canada': 'North America', 'Mexico': 'North America',
        'Indonesia': 'Asia', 'India': 'Asia', 'Japan': 'Asia', 'Thailand': 'Asia',
        'United Kingdom': 'Europe', 'France': 'Europe', 'Germany': 'Europe'
    }
    return mapping.get(country, 'Others')

# inisiasi Transformasi Data Inference
inf_data['continent'] = inf_data['country'].apply(country_to_continent)
inf_data_final = inf_data.drop(columns=['country'])

# inisiasi Eksekusi Prediksi
pred_result = model_inf.predict(inf_data_final)
pred_proba = model_inf.predict_proba(inf_data_final)[:, 1]

# inisiasi Prediksi dengan Model Tuned
final_pred = final_model.predict(inf_data_final)
final_proba = final_model.predict_proba(inf_data_final)[:, 1]

#inisiasi untuk menampilkan Hasil Akhir modelling 2

print("=== HASIL PERBANDINGAN ANTARA USER 1 DAN 2 ===")
for i in range(len(inf_data_final)):
    label = "AKAN MEMBELI" if pred_result[i] == 1 else "TIDAK MEMBELI"
    print(f"User {i+1}: {label} (Probabilitas: {pred_proba[i]:.2f})")
    
    
# Inisiasi untuk menampilkan hasil
print("=== HASIL PREDIKSI SEBERAPA POTENSIAL, PEMBELI AKAN MEMBELI DIMASA YANG AKAN DATANG (TUNED) ===")
for i in range(len(inf_data_final)):
    label = "PEMBELI POTENSIAL" if final_pred[i] == 1 else "BUKAN PEMBELI"
    print(f"User {i+1}: {label}")
    print(f"Confidence Score: {final_proba[i]:.2f}")
    print("-" * 35)

=== Hasil Prediksi Modeling 1 ===
User 1: Akan Membeli (Probabilitas: 0.78)
User 2: Akan Membeli (Probabilitas: 0.78)
User 1: AKAN MEMBELI (Probabilitas: 0.88)
User 2: TIDAK MEMBELI (Probabilitas: 0.26)
=== HASIL PREDIKSI MODEL FINAL (TUNED) ===
User 1: PEMBELI POTENSIAL
Confidence Score: 0.87
-----------------------------------
User 2: PEMBELI POTENSIAL
Confidence Score: 0.64
-----------------------------------


Model akhir berhasil memprediksi bahwa user 1 akan melakukan pembelian di masa akan mendatang dengan tingkat keyakinan 88%, sedangkan user 2 diprediksi tidak akan melakukan pembelian di masa akan mendatang dengan tingat keyakinan 26%.